In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from category_encoders.leave_one_out import LeaveOneOutEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

import pickle

In [ ]:
df=pd.read_csv("crop_yield.csv")
df.columns

Index(['Region', 'Soil_Type', 'Crop', 'Rainfall_mm', 'Temperature_Celsius',
       'Fertilizer_Used', 'Irrigation_Used', 'Weather_Condition',
       'Days_to_Harvest', 'Yield_tons_per_hectare'],
      dtype='object')

In [3]:
df=df.rename(columns={
    "Temperature_Celsius":"temperature",
    "Rainfall_mm":"rainfall",
    "Soil_Type":"soil_type",
    "Weather_Condition":"weather_condition",
    "Yield_tons_per_hectare":"yield"
})
df.columns

Index(['Region', 'soil_type', 'Crop', 'rainfall', 'temperature',
       'Fertilizer_Used', 'Irrigation_Used', 'weather_condition',
       'Days_to_Harvest', 'yield'],
      dtype='object')

In [4]:
x=df.drop("yield",axis=1)
y=df["yield"]
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


In [5]:
x_train

,Region,soil_type,Crop,rainfall,temperature,Fertilizer_Used,Irrigation_Used,weather_condition,Days_to_Harvest
566853,North,Sandy,Soybean,887.383906,33.050569,True,True,Cloudy,79
382311,South,Clay,Soybean,179.188444,24.699056,False,False,Rainy,88
241519,South,Silt,Wheat,649.310754,30.057577,False,False,Cloudy,120
719220,North,Loam,Cotton,221.489100,19.508065,False,True,Cloudy,101
905718,East,Peaty,Rice,737.016449,18.233438,True,False,Rainy,88
...,...,...,...,...,...,...,...,...,...
259178,North,Chalky,Rice,200.893070,26.934797,False,True,Cloudy,102
365838,North,Peaty,Cotton,600.707134,29.819791,True,False,Sunny,81
131932,North,Chalky,Cotton,455.647331,35.877561,False,True,Rainy,86
671155,North,Loam,Cotton,658.816440,19.619801,True,True,Cloudy,89


In [6]:
x_train_ohe=pd.get_dummies(x_train,columns=["soil_type","weather_condition","Crop","Region"],drop_first=True)
x_test_ohe=pd.get_dummies(x_test,columns=["soil_type","weather_condition","Crop","Region"],drop_first=True)
x_train_ohe
#one hot encoding

,rainfall,temperature,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,soil_type_Clay,soil_type_Loam,soil_type_Peaty,soil_type_Sandy,soil_type_Silt,weather_condition_Rainy,weather_condition_Sunny,Crop_Cotton,Crop_Maize,Crop_Rice,Crop_Soybean,Crop_Wheat,Region_North,Region_South,Region_West
566853,887.383906,33.050569,True,True,79,False,False,False,True,False,False,False,False,False,False,True,False,True,False,False
382311,179.188444,24.699056,False,False,88,True,False,False,False,False,True,False,False,False,False,True,False,False,True,False
241519,649.310754,30.057577,False,False,120,False,False,False,False,True,False,False,False,False,False,False,True,False,True,False
719220,221.489100,19.508065,False,True,101,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False
905718,737.016449,18.233438,True,False,88,False,False,True,False,False,True,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259178,200.893070,26.934797,False,True,102,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False
365838,600.707134,29.819791,True,False,81,False,False,True,False,False,False,True,True,False,False,False,False,True,False,False
131932,455.647331,35.877561,False,True,86,False,False,False,False,False,True,False,True,False,False,False,False,True,False,False
671155,658.816440,19.619801,True,True,89,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False


In [7]:
x_train_ohe, x_test_ohe = x_train_ohe.align(x_test_ohe, join='outer', axis=1, fill_value=0)

In [8]:
leoe_encoder=LeaveOneOutEncoder(cols=["soil_type","weather_condition","Crop","Region"])
x_train_leoe=leoe_encoder.fit_transform(x_train,y_train)
x_test_leoe=leoe_encoder.transform(x_test)
x_train_leoe

,Region,soil_type,Crop,rainfall,temperature,Fertilizer_Used,Irrigation_Used,weather_condition,Days_to_Harvest
566853,4.652371,4.646157,4.652244,887.383906,33.050569,True,True,4.647582,79
382311,4.648410,4.645563,4.652300,179.188444,24.699056,False,False,4.648543,88
241519,4.648391,4.647616,4.654280,649.310754,30.057577,False,False,4.647596,120
719220,4.652398,4.654132,4.649399,221.489100,19.508065,False,True,4.647603,101
905718,4.645159,4.650633,4.652075,737.016449,18.233438,True,False,4.648528,88
...,...,...,...,...,...,...,...,...,...
259178,4.652398,4.650057,4.652089,200.893070,26.934797,False,True,4.647602,102
365838,4.652388,4.650632,4.649384,600.707134,29.819791,True,False,4.650928,81
131932,4.652394,4.650050,4.649392,455.647331,35.877561,False,True,4.648532,86
671155,4.652382,4.654108,4.649375,658.816440,19.619801,True,True,4.647590,89


In [9]:
len(x_train_leoe.columns)

9

In [10]:
with open("leoe_encoder.pkl","wb") as f:
    pickle.dump(leoe_encoder,f)

In [11]:
print(len(df),len(df.columns))
print(df.columns)
df_clean=df.dropna(axis=0)
df_clean=df_clean.dropna(axis=1)
df_clean=df_clean.dropna(subset=["temperature","rainfall"])
print(len(df_clean),len(df_clean.columns))

1000000 10
Index(['Region', 'soil_type', 'Crop', 'rainfall', 'temperature',
       'Fertilizer_Used', 'Irrigation_Used', 'weather_condition',
       'Days_to_Harvest', 'yield'],
      dtype='object')
1000000 10


In [12]:
scalar_std = StandardScaler()

# Select only the numeric columns you want to scale
x_train_ohe_scaled = scalar_std.fit_transform(x_train_ohe[["temperature", "rainfall"]])
x_test_ohe_scaled = scalar_std.transform(x_test_ohe[["temperature", "rainfall"]])

In [13]:
x_train_ohe_scaled

array([[ 0.76726829,  1.29921084],
       [-0.38947184, -1.42674651],
       [ 0.35271903,  0.38282928],
       ...,
       [ 1.15882554, -0.36261214],
       [-1.09298252,  0.41941819],
       [-0.75449531,  0.44903755]], shape=(800000, 2))

In [14]:
scalar_mn=MinMaxScaler()
x_train_ohe_scaled_mn = scalar_mn.fit_transform(x_train_ohe[["temperature", "rainfall"]])
x_test_ohe_scaled_mn= scalar_mn.transform(x_test_ohe[["temperature", "rainfall"]])

In [15]:
x_train_ohe_scaled_mn

array([[0.72202239, 0.87487273],
       [0.3879613 , 0.08798644],
       [0.60230252, 0.61034618],
       ...,
       [0.83510228, 0.39516393],
       [0.18479074, 0.62090809],
       [0.28254425, 0.62945814]], shape=(800000, 2))

In [16]:
numerical_columns=["temperature","rainfall"]

In [17]:
x_train_ohe_scaled_mn_df=pd.DataFrame(x_train_ohe_scaled_mn, columns=numerical_columns,index=x_train_ohe.index)
print(type(x_train_ohe_scaled_mn_df))
x_train_ohe_scaled_mn_df
#converts into dataframe

<class 'pandas.core.frame.DataFrame'>


,temperature,rainfall
566853,0.722022,0.874873
382311,0.387961,0.087986
241519,0.602303,0.610346
719220,0.180321,0.134987
905718,0.129336,0.707797
...,...,...
259178,0.477391,0.112103
365838,0.592791,0.556342
131932,0.835102,0.395164
671155,0.184791,0.620908


In [18]:
x_test_ohe_scaled_mn_df=pd.DataFrame(x_test_ohe_scaled_mn, columns=numerical_columns,index=x_test_ohe.index)
print(type(x_test_ohe_scaled_mn_df))
x_test_ohe_scaled_mn_df
#converts into dataframe

<class 'pandas.core.frame.DataFrame'>


,temperature,rainfall
987231,0.355034,0.683173
79954,0.322835,0.845118
567130,0.360804,0.780092
500891,0.075807,0.115129
55399,0.136115,0.456143
...,...,...
90245,0.831907,0.990436
639296,0.780282,0.018642
311939,0.627617,0.186521
324459,0.538636,0.636985


In [19]:
#join the scaled values with other columns - for this we use concat method
final_train_ohe_scaled_mm=pd.concat([x_train_ohe_scaled_mn_df, x_train_ohe.drop(columns=numerical_columns)], axis=1)
final_train_ohe_scaled_mm

,temperature,rainfall,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,soil_type_Clay,soil_type_Loam,soil_type_Peaty,soil_type_Sandy,soil_type_Silt,weather_condition_Rainy,weather_condition_Sunny,Crop_Cotton,Crop_Maize,Crop_Rice,Crop_Soybean,Crop_Wheat,Region_North,Region_South,Region_West
566853,0.722022,0.874873,True,True,79,False,False,False,True,False,False,False,False,False,False,True,False,True,False,False
382311,0.387961,0.087986,False,False,88,True,False,False,False,False,True,False,False,False,False,True,False,False,True,False
241519,0.602303,0.610346,False,False,120,False,False,False,False,True,False,False,False,False,False,False,True,False,True,False
719220,0.180321,0.134987,False,True,101,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False
905718,0.129336,0.707797,True,False,88,False,False,True,False,False,True,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259178,0.477391,0.112103,False,True,102,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False
365838,0.592791,0.556342,True,False,81,False,False,True,False,False,False,True,True,False,False,False,False,True,False,False
131932,0.835102,0.395164,False,True,86,False,False,False,False,False,True,False,True,False,False,False,False,True,False,False
671155,0.184791,0.620908,True,True,89,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False


In [20]:
#join the scaled values with other columns - for this we use concat method
final_test_ohe_scaled_mm=pd.concat([x_test_ohe_scaled_mn_df, x_test_ohe.drop(columns=numerical_columns)], axis=1)
final_test_ohe_scaled_mm

,temperature,rainfall,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,soil_type_Clay,soil_type_Loam,soil_type_Peaty,soil_type_Sandy,soil_type_Silt,weather_condition_Rainy,weather_condition_Sunny,Crop_Cotton,Crop_Maize,Crop_Rice,Crop_Soybean,Crop_Wheat,Region_North,Region_South,Region_West
987231,0.355034,0.683173,False,False,120,False,False,False,False,True,False,True,True,False,False,False,False,False,False,True
79954,0.322835,0.845118,False,False,78,False,False,False,False,False,True,False,True,False,False,False,False,True,False,False
567130,0.360804,0.780092,True,True,140,False,False,False,True,False,True,False,False,False,False,False,False,True,False,False
500891,0.075807,0.115129,False,True,96,False,False,False,False,False,False,True,True,False,False,False,False,False,False,True
55399,0.136115,0.456143,False,True,65,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90245,0.831907,0.990436,False,True,85,False,False,False,True,False,False,True,False,False,False,False,True,True,False,False
639296,0.780282,0.018642,True,True,82,False,False,False,False,True,True,False,False,True,False,False,False,False,False,False
311939,0.627617,0.186521,True,True,149,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True
324459,0.538636,0.636985,False,False,68,False,True,False,False,False,True,False,False,False,False,True,False,True,False,False


In [21]:
len(final_train_ohe_scaled_mm.columns)

20

In [22]:
final_train_ohe_scaled_mm["split"]="train"
final_test_ohe_scaled_mm["split"]="test"

In [23]:
y_train

566853    8.285534
382311    0.760165
241519    4.563030
719220    2.872347
905718    4.809969
            ...   
259178    2.924335
365838    4.909864
131932    3.793084
671155    6.096108
121958    5.184939
Name: yield, Length: 800000, dtype: float64

y_train

In [24]:
final_train_ohe_scaled_mm

,temperature,rainfall,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,soil_type_Clay,soil_type_Loam,soil_type_Peaty,soil_type_Sandy,soil_type_Silt,...,weather_condition_Sunny,Crop_Cotton,Crop_Maize,Crop_Rice,Crop_Soybean,Crop_Wheat,Region_North,Region_South,Region_West,split
566853,0.722022,0.874873,True,True,79,False,False,False,True,False,...,False,False,False,False,True,False,True,False,False,train
382311,0.387961,0.087986,False,False,88,True,False,False,False,False,...,False,False,False,False,True,False,False,True,False,train
241519,0.602303,0.610346,False,False,120,False,False,False,False,True,...,False,False,False,False,False,True,False,True,False,train
719220,0.180321,0.134987,False,True,101,False,True,False,False,False,...,False,True,False,False,False,False,True,False,False,train
905718,0.129336,0.707797,True,False,88,False,False,True,False,False,...,False,False,False,True,False,False,False,False,False,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259178,0.477391,0.112103,False,True,102,False,False,False,False,False,...,False,False,False,True,False,False,True,False,False,train
365838,0.592791,0.556342,True,False,81,False,False,True,False,False,...,True,True,False,False,False,False,True,False,False,train
131932,0.835102,0.395164,False,True,86,False,False,False,False,False,...,False,True,False,False,False,False,True,False,False,train
671155,0.184791,0.620908,True,True,89,False,True,False,False,False,...,False,True,False,False,False,False,True,False,False,train


In [25]:
final_ohe_scaled_mm = pd.concat([final_train_ohe_scaled_mm, final_test_ohe_scaled_mm], axis=0)


In [26]:
final_ohe_scaled_mm.columns

Index(['temperature', 'rainfall', 'Fertilizer_Used', 'Irrigation_Used',
       'Days_to_Harvest', 'soil_type_Clay', 'soil_type_Loam',
       'soil_type_Peaty', 'soil_type_Sandy', 'soil_type_Silt',
       'weather_condition_Rainy', 'weather_condition_Sunny', 'Crop_Cotton',
       'Crop_Maize', 'Crop_Rice', 'Crop_Soybean', 'Crop_Wheat', 'Region_North',
       'Region_South', 'Region_West', 'split'],
      dtype='object')

In [27]:
final_y = pd.concat([y_train, y_test], axis=0)

In [28]:
print("Columns in final_ohe_scaled_mm:", final_ohe_scaled_mm.columns.tolist())



Columns in final_ohe_scaled_mm: ['temperature', 'rainfall', 'Fertilizer_Used', 'Irrigation_Used', 'Days_to_Harvest', 'soil_type_Clay', 'soil_type_Loam', 'soil_type_Peaty', 'soil_type_Sandy', 'soil_type_Silt', 'weather_condition_Rainy', 'weather_condition_Sunny', 'Crop_Cotton', 'Crop_Maize', 'Crop_Rice', 'Crop_Soybean', 'Crop_Wheat', 'Region_North', 'Region_South', 'Region_West', 'split']


In [29]:


final_df = pd.concat([final_ohe_scaled_mm, final_y], axis=1)


In [30]:
final_ohe_scaled_mm.to_csv("datasets/agriyield/final_ohe_scaled_mm.csv",index="false")

In [31]:
df_loaded=pd.read_csv("datasets/agriyield/final_ohe_scaled_mm.csv")

In [32]:
print(df_loaded.columns)


Index(['Unnamed: 0', 'temperature', 'rainfall', 'Fertilizer_Used',
       'Irrigation_Used', 'Days_to_Harvest', 'soil_type_Clay',
       'soil_type_Loam', 'soil_type_Peaty', 'soil_type_Sandy',
       'soil_type_Silt', 'weather_condition_Rainy', 'weather_condition_Sunny',
       'Crop_Cotton', 'Crop_Maize', 'Crop_Rice', 'Crop_Soybean', 'Crop_Wheat',
       'Region_North', 'Region_South', 'Region_West', 'split'],
      dtype='object')


In [33]:
# Assuming 80% train, 20% test
train_size = int(0.8 * len(df_loaded))
df_loaded['split'] = ['train'] * train_size + ['test'] * (len(df_loaded) - train_size)



In [34]:
train_loaded_df = df_loaded[df_loaded['split'] == 'train'].drop(columns=['split'])
test_loaded_df = df_loaded[df_loaded['split'] == 'test'].drop(columns=['split'])


In [35]:
test_loaded_df

,Unnamed: 0,temperature,rainfall,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,soil_type_Clay,soil_type_Loam,soil_type_Peaty,soil_type_Sandy,...,weather_condition_Rainy,weather_condition_Sunny,Crop_Cotton,Crop_Maize,Crop_Rice,Crop_Soybean,Crop_Wheat,Region_North,Region_South,Region_West
800000,987231,0.355034,0.683173,False,False,120,False,False,False,False,...,False,True,True,False,False,False,False,False,False,True
800001,79954,0.322835,0.845118,False,False,78,False,False,False,False,...,True,False,True,False,False,False,False,True,False,False
800002,567130,0.360804,0.780092,True,True,140,False,False,False,True,...,True,False,False,False,False,False,False,True,False,False
800003,500891,0.075807,0.115129,False,True,96,False,False,False,False,...,False,True,True,False,False,False,False,False,False,True
800004,55399,0.136115,0.456143,False,True,65,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,90245,0.831907,0.990436,False,True,85,False,False,False,True,...,False,True,False,False,False,False,True,True,False,False
999996,639296,0.780282,0.018642,True,True,82,False,False,False,False,...,True,False,False,True,False,False,False,False,False,False
999997,311939,0.627617,0.186521,True,True,149,False,False,True,False,...,False,False,False,True,False,False,False,False,False,True
999998,324459,0.538636,0.636985,False,False,68,False,True,False,False,...,True,False,False,False,False,True,False,True,False,False


In [36]:
final_train_ohe_scaled_mm["rainfall_temp ratio"]=final_train_ohe_scaled_mm["rainfall"]/final_train_ohe_scaled_mm["temperature"]

In [37]:
final_train_ohe_scaled_mm

,temperature,rainfall,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,soil_type_Clay,soil_type_Loam,soil_type_Peaty,soil_type_Sandy,soil_type_Silt,...,Crop_Cotton,Crop_Maize,Crop_Rice,Crop_Soybean,Crop_Wheat,Region_North,Region_South,Region_West,split,rainfall_temp ratio
566853,0.722022,0.874873,True,True,79,False,False,False,True,False,...,False,False,False,True,False,True,False,False,train,1.211698
382311,0.387961,0.087986,False,False,88,True,False,False,False,False,...,False,False,False,True,False,False,True,False,train,0.226792
241519,0.602303,0.610346,False,False,120,False,False,False,False,True,...,False,False,False,False,True,False,True,False,train,1.013355
719220,0.180321,0.134987,False,True,101,False,True,False,False,False,...,True,False,False,False,False,True,False,False,train,0.748593
905718,0.129336,0.707797,True,False,88,False,False,True,False,False,...,False,False,True,False,False,False,False,False,train,5.472542
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259178,0.477391,0.112103,False,True,102,False,False,False,False,False,...,False,False,True,False,False,True,False,False,train,0.234824
365838,0.592791,0.556342,True,False,81,False,False,True,False,False,...,True,False,False,False,False,True,False,False,train,0.938513
131932,0.835102,0.395164,False,True,86,False,False,False,False,False,...,True,False,False,False,False,True,False,False,train,0.473192
671155,0.184791,0.620908,True,True,89,False,True,False,False,False,...,True,False,False,False,False,True,False,False,train,3.360061


In [38]:
#converting standard scalar ohe to dataframe
x_train_ohe_scaled_df=pd.DataFrame(x_train_ohe_scaled, columns=numerical_columns,index=x_train_ohe.index)
print(type(x_train_ohe_scaled_df))
x_train_ohe_scaled_df
#converts into dataframe

<class 'pandas.core.frame.DataFrame'>


,temperature,rainfall
566853,0.767268,1.299211
382311,-0.389472,-1.426747
241519,0.352719,0.382829
719220,-1.108459,-1.263925
905718,-1.285003,0.720423
...,...,...
259178,-0.079807,-1.343202
365838,0.319784,0.195746
131932,1.158826,-0.362612
671155,-1.092983,0.419418


In [39]:
x_test_ohe_scaled_df=pd.DataFrame(x_test_ohe_scaled, columns=numerical_columns,index=x_test_ohe.index)
print(type(x_test_ohe_scaled_df))
x_test_ohe_scaled_df
#converts into dataframe

<class 'pandas.core.frame.DataFrame'>


,temperature,rainfall
987231,-0.503488,0.635117
79954,-0.614983,1.196133
567130,-0.483508,0.970870
500891,-1.470356,-1.332717
55399,-1.261531,-0.151367
...,...,...
90245,1.147761,1.699549
639296,0.969002,-1.666972
311939,0.440376,-1.085401
324459,0.132263,0.475114


In [40]:
#join the scaled values with other columns - for this we use concat method
final_train_ohe_scaled=pd.concat([x_train_ohe_scaled_df, x_train_ohe.drop(columns=numerical_columns)], axis=1)
final_train_ohe_scaled

,temperature,rainfall,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,soil_type_Clay,soil_type_Loam,soil_type_Peaty,soil_type_Sandy,soil_type_Silt,weather_condition_Rainy,weather_condition_Sunny,Crop_Cotton,Crop_Maize,Crop_Rice,Crop_Soybean,Crop_Wheat,Region_North,Region_South,Region_West
566853,0.767268,1.299211,True,True,79,False,False,False,True,False,False,False,False,False,False,True,False,True,False,False
382311,-0.389472,-1.426747,False,False,88,True,False,False,False,False,True,False,False,False,False,True,False,False,True,False
241519,0.352719,0.382829,False,False,120,False,False,False,False,True,False,False,False,False,False,False,True,False,True,False
719220,-1.108459,-1.263925,False,True,101,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False
905718,-1.285003,0.720423,True,False,88,False,False,True,False,False,True,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259178,-0.079807,-1.343202,False,True,102,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False
365838,0.319784,0.195746,True,False,81,False,False,True,False,False,False,True,True,False,False,False,False,True,False,False
131932,1.158826,-0.362612,False,True,86,False,False,False,False,False,True,False,True,False,False,False,False,True,False,False
671155,-1.092983,0.419418,True,True,89,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False


In [41]:
len(final_train_ohe_scaled.columns)

20

In [42]:
final_test_ohe_scaled=pd.concat([x_test_ohe_scaled_df, x_test_ohe.drop(columns=numerical_columns)], axis=1)
final_test_ohe_scaled

,temperature,rainfall,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,soil_type_Clay,soil_type_Loam,soil_type_Peaty,soil_type_Sandy,soil_type_Silt,weather_condition_Rainy,weather_condition_Sunny,Crop_Cotton,Crop_Maize,Crop_Rice,Crop_Soybean,Crop_Wheat,Region_North,Region_South,Region_West
987231,-0.503488,0.635117,False,False,120,False,False,False,False,True,False,True,True,False,False,False,False,False,False,True
79954,-0.614983,1.196133,False,False,78,False,False,False,False,False,True,False,True,False,False,False,False,True,False,False
567130,-0.483508,0.970870,True,True,140,False,False,False,True,False,True,False,False,False,False,False,False,True,False,False
500891,-1.470356,-1.332717,False,True,96,False,False,False,False,False,False,True,True,False,False,False,False,False,False,True
55399,-1.261531,-0.151367,False,True,65,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90245,1.147761,1.699549,False,True,85,False,False,False,True,False,False,True,False,False,False,False,True,True,False,False
639296,0.969002,-1.666972,True,True,82,False,False,False,False,True,True,False,False,True,False,False,False,False,False,False
311939,0.440376,-1.085401,True,True,149,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True
324459,0.132263,0.475114,False,False,68,False,True,False,False,False,True,False,False,False,False,True,False,True,False,False


In [43]:
final_train_ohe_scaled["split"]="train"
final_test_ohe_scaled["split"]="test"

In [44]:
final_train_ohe_scaled


,temperature,rainfall,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,soil_type_Clay,soil_type_Loam,soil_type_Peaty,soil_type_Sandy,soil_type_Silt,...,weather_condition_Sunny,Crop_Cotton,Crop_Maize,Crop_Rice,Crop_Soybean,Crop_Wheat,Region_North,Region_South,Region_West,split
566853,0.767268,1.299211,True,True,79,False,False,False,True,False,...,False,False,False,False,True,False,True,False,False,train
382311,-0.389472,-1.426747,False,False,88,True,False,False,False,False,...,False,False,False,False,True,False,False,True,False,train
241519,0.352719,0.382829,False,False,120,False,False,False,False,True,...,False,False,False,False,False,True,False,True,False,train
719220,-1.108459,-1.263925,False,True,101,False,True,False,False,False,...,False,True,False,False,False,False,True,False,False,train
905718,-1.285003,0.720423,True,False,88,False,False,True,False,False,...,False,False,False,True,False,False,False,False,False,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259178,-0.079807,-1.343202,False,True,102,False,False,False,False,False,...,False,False,False,True,False,False,True,False,False,train
365838,0.319784,0.195746,True,False,81,False,False,True,False,False,...,True,True,False,False,False,False,True,False,False,train
131932,1.158826,-0.362612,False,True,86,False,False,False,False,False,...,False,True,False,False,False,False,True,False,False,train
671155,-1.092983,0.419418,True,True,89,False,True,False,False,False,...,False,True,False,False,False,False,True,False,False,train


In [45]:
final_ohe_scaled=pd.concat([final_train_ohe_scaled, final_test_ohe_scaled],axis=0)

In [46]:
final_ohe_scaled

,temperature,rainfall,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,soil_type_Clay,soil_type_Loam,soil_type_Peaty,soil_type_Sandy,soil_type_Silt,...,weather_condition_Sunny,Crop_Cotton,Crop_Maize,Crop_Rice,Crop_Soybean,Crop_Wheat,Region_North,Region_South,Region_West,split
566853,0.767268,1.299211,True,True,79,False,False,False,True,False,...,False,False,False,False,True,False,True,False,False,train
382311,-0.389472,-1.426747,False,False,88,True,False,False,False,False,...,False,False,False,False,True,False,False,True,False,train
241519,0.352719,0.382829,False,False,120,False,False,False,False,True,...,False,False,False,False,False,True,False,True,False,train
719220,-1.108459,-1.263925,False,True,101,False,True,False,False,False,...,False,True,False,False,False,False,True,False,False,train
905718,-1.285003,0.720423,True,False,88,False,False,True,False,False,...,False,False,False,True,False,False,False,False,False,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90245,1.147761,1.699549,False,True,85,False,False,False,True,False,...,True,False,False,False,False,True,True,False,False,test
639296,0.969002,-1.666972,True,True,82,False,False,False,False,True,...,False,False,True,False,False,False,False,False,False,test
311939,0.440376,-1.085401,True,True,149,False,False,True,False,False,...,False,False,True,False,False,False,False,False,True,test
324459,0.132263,0.475114,False,False,68,False,True,False,False,False,...,False,False,False,False,True,False,True,False,False,test


In [47]:
final_ohe_scaled = final_ohe_scaled_mm.reset_index(drop=True)


final_df1 = pd.concat([final_ohe_scaled, final_y], axis=1)
final_df1


,temperature,rainfall,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,soil_type_Clay,soil_type_Loam,soil_type_Peaty,soil_type_Sandy,soil_type_Silt,...,Crop_Cotton,Crop_Maize,Crop_Rice,Crop_Soybean,Crop_Wheat,Region_North,Region_South,Region_West,split,yield
0,0.722022,0.874873,True,True,79,False,False,False,True,False,...,False,False,False,True,False,True,False,False,train,6.555816
1,0.387961,0.087986,False,False,88,True,False,False,False,False,...,False,False,False,True,False,False,True,False,train,8.527341
2,0.602303,0.610346,False,False,120,False,False,False,False,True,...,False,False,False,False,True,False,True,False,train,1.127443
3,0.180321,0.134987,False,True,101,False,True,False,False,False,...,True,False,False,False,False,True,False,False,train,6.517573
4,0.129336,0.707797,True,False,88,False,False,True,False,False,...,False,False,True,False,False,False,False,False,train,7.248251
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,0.831907,0.990436,False,True,85,False,False,False,True,False,...,False,False,False,False,True,True,False,False,test,1.347586
999996,0.780282,0.018642,True,True,82,False,False,False,False,True,...,False,True,False,False,False,False,False,False,test,7.311594
999997,0.627617,0.186521,True,True,149,False,False,True,False,False,...,False,True,False,False,False,False,False,True,test,5.763182
999998,0.538636,0.636985,False,False,68,False,True,False,False,False,...,False,False,False,True,False,True,False,False,test,2.070159


In [48]:
final_ohe_scaled.to_csv("datasets/agriyield/final_ohe_scaled.csv",index="false")

In [49]:
df_loaded1=pd.read_csv("datasets/agriyield/final_ohe_scaled.csv")

In [50]:
# Assuming 80% train, 20% test
train_size = int(0.8 * len(df_loaded1))
df_loaded1['split'] = ['train'] * train_size + ['test'] * (len(df_loaded1) - train_size)



In [51]:
train_loaded_df2=df_loaded1[df_loaded1["split"]=="train"].drop(columns=["split"])
test_loaded_df2=df_loaded1[df_loaded1["split"]=="test"].drop(columns=["split"])

In [52]:
scalar_std = StandardScaler()

# Select only the numeric columns you want to scale-leaveoneoutencoding
x_train_leoe_scaled = scalar_std.fit_transform(x_train_leoe[["temperature", "rainfall"]])
x_test_leoe_scaled = scalar_std.transform(x_test_leoe[["temperature", "rainfall"]])

In [53]:
x_train_leoe_scaled


array([[ 0.76726829,  1.29921084],
       [-0.38947184, -1.42674651],
       [ 0.35271903,  0.38282928],
       ...,
       [ 1.15882554, -0.36261214],
       [-1.09298252,  0.41941819],
       [-0.75449531,  0.44903755]], shape=(800000, 2))

In [54]:
#converting standard scalar ohe to dataframe
x_train_leoe_scaled_df=pd.DataFrame(x_train_leoe_scaled, columns=numerical_columns,index=x_train_leoe.index)
print(type(x_train_leoe_scaled_df))
x_train_leoe_scaled_df
#converts into dataframe

<class 'pandas.core.frame.DataFrame'>


,temperature,rainfall
566853,0.767268,1.299211
382311,-0.389472,-1.426747
241519,0.352719,0.382829
719220,-1.108459,-1.263925
905718,-1.285003,0.720423
...,...,...
259178,-0.079807,-1.343202
365838,0.319784,0.195746
131932,1.158826,-0.362612
671155,-1.092983,0.419418


In [55]:
#converting standard scalar ohe to dataframe
x_test_leoe_scaled_df=pd.DataFrame(x_test_leoe_scaled, columns=numerical_columns,index=x_test_leoe.index)
print(type(x_test_leoe_scaled_df))
x_test_leoe_scaled_df
#converts into dataframe

<class 'pandas.core.frame.DataFrame'>


,temperature,rainfall
987231,-0.503488,0.635117
79954,-0.614983,1.196133
567130,-0.483508,0.970870
500891,-1.470356,-1.332717
55399,-1.261531,-0.151367
...,...,...
90245,1.147761,1.699549
639296,0.969002,-1.666972
311939,0.440376,-1.085401
324459,0.132263,0.475114


In [56]:
print(x_train_leoe_scaled_df.shape)
print(x_train_leoe.drop(columns=numerical_columns).shape)


(800000, 2)
(800000, 7)


In [57]:
#join the scaled values with other columns - for this we use concat method
final_train_leoe_scaled=pd.concat([x_train_leoe_scaled_df, x_train_leoe.drop(columns=numerical_columns)], axis=1)
final_train_leoe_scaled

,temperature,rainfall,Region,soil_type,Crop,Fertilizer_Used,Irrigation_Used,weather_condition,Days_to_Harvest
566853,0.767268,1.299211,4.652371,4.646157,4.652244,True,True,4.647582,79
382311,-0.389472,-1.426747,4.648410,4.645563,4.652300,False,False,4.648543,88
241519,0.352719,0.382829,4.648391,4.647616,4.654280,False,False,4.647596,120
719220,-1.108459,-1.263925,4.652398,4.654132,4.649399,False,True,4.647603,101
905718,-1.285003,0.720423,4.645159,4.650633,4.652075,True,False,4.648528,88
...,...,...,...,...,...,...,...,...,...
259178,-0.079807,-1.343202,4.652398,4.650057,4.652089,False,True,4.647602,102
365838,0.319784,0.195746,4.652388,4.650632,4.649384,True,False,4.650928,81
131932,1.158826,-0.362612,4.652394,4.650050,4.649392,False,True,4.648532,86
671155,-1.092983,0.419418,4.652382,4.654108,4.649375,True,True,4.647590,89


In [58]:
#join the scaled values with other columns - for this we use concat method
final_test_leoe_scaled=pd.concat([x_test_leoe_scaled_df, x_test_leoe.drop(columns=numerical_columns)], axis=1)
final_test_leoe_scaled.head()

,temperature,rainfall,Region,soil_type,Crop,Fertilizer_Used,Irrigation_Used,weather_condition,Days_to_Harvest
987231,-0.503488,0.635117,4.650130,4.647616,4.649386,False,False,4.650929,120
79954,-0.614983,1.196133,4.652389,4.650044,4.649386,False,False,4.648529,78
567130,-0.483508,0.970870,4.652389,4.646184,4.645186,True,True,4.648529,140
500891,-1.470356,-1.332717,4.650130,4.650044,4.649386,False,True,4.650929,96
55399,-1.261531,-0.151367,4.645160,4.647616,4.652076,False,True,4.647596,65


In [59]:
final_train_leoe_scaled["split"]="train"
final_test_leoe_scaled["split"]="test"

In [60]:
final_train_leoe_scaled


,temperature,rainfall,Region,soil_type,Crop,Fertilizer_Used,Irrigation_Used,weather_condition,Days_to_Harvest,split
566853,0.767268,1.299211,4.652371,4.646157,4.652244,True,True,4.647582,79,train
382311,-0.389472,-1.426747,4.648410,4.645563,4.652300,False,False,4.648543,88,train
241519,0.352719,0.382829,4.648391,4.647616,4.654280,False,False,4.647596,120,train
719220,-1.108459,-1.263925,4.652398,4.654132,4.649399,False,True,4.647603,101,train
905718,-1.285003,0.720423,4.645159,4.650633,4.652075,True,False,4.648528,88,train
...,...,...,...,...,...,...,...,...,...,...
259178,-0.079807,-1.343202,4.652398,4.650057,4.652089,False,True,4.647602,102,train
365838,0.319784,0.195746,4.652388,4.650632,4.649384,True,False,4.650928,81,train
131932,1.158826,-0.362612,4.652394,4.650050,4.649392,False,True,4.648532,86,train
671155,-1.092983,0.419418,4.652382,4.654108,4.649375,True,True,4.647590,89,train


In [61]:
final_leoe_scaled=pd.concat([final_train_leoe_scaled, final_test_leoe_scaled],axis=0)

In [62]:
final_leoe_scaled = final_leoe_scaled.reset_index(drop=True)


final_df2 = pd.concat([final_leoe_scaled, final_y], axis=1)


In [63]:
final_leoe_scaled

,temperature,rainfall,Region,soil_type,Crop,Fertilizer_Used,Irrigation_Used,weather_condition,Days_to_Harvest,split
0,0.767268,1.299211,4.652371,4.646157,4.652244,True,True,4.647582,79,train
1,-0.389472,-1.426747,4.648410,4.645563,4.652300,False,False,4.648543,88,train
2,0.352719,0.382829,4.648391,4.647616,4.654280,False,False,4.647596,120,train
3,-1.108459,-1.263925,4.652398,4.654132,4.649399,False,True,4.647603,101,train
4,-1.285003,0.720423,4.645159,4.650633,4.652075,True,False,4.648528,88,train
...,...,...,...,...,...,...,...,...,...,...
999995,1.147761,1.699549,4.652389,4.646184,4.654279,False,True,4.650929,85,test
999996,0.969002,-1.666972,4.645160,4.647616,4.640918,True,True,4.648529,82,test
999997,0.440376,-1.085401,4.650130,4.650634,4.640918,True,True,4.647596,149,test
999998,0.132263,0.475114,4.652389,4.654119,4.652271,False,False,4.648529,68,test


In [64]:
final_leoe_scaled.to_csv("datasets/agriyield/final_leoe_scaled.csv",index="false")

In [65]:
df_loaded2=pd.read_csv("datasets/agriyield/final_leoe_scaled.csv")

In [66]:
train_loaded_df3=df_loaded2[df_loaded2["split"]=="train"].drop(columns=["split"])
test_loaded_df3=df_loaded2[df_loaded2["split"]=="test"].drop(columns=["split"])

In [67]:
scalar_mn=MinMaxScaler()
x_train_leoe_scaled_mn = scalar_mn.fit_transform(x_train_leoe[["temperature", "rainfall"]])
x_test_leoe_scaled_mn= scalar_mn.transform(x_test_leoe[["temperature", "rainfall"]])

In [68]:
x_train_leoe_scaled_mn


array([[0.72202239, 0.87487273],
       [0.3879613 , 0.08798644],
       [0.60230252, 0.61034618],
       ...,
       [0.83510228, 0.39516393],
       [0.18479074, 0.62090809],
       [0.28254425, 0.62945814]], shape=(800000, 2))

In [69]:
x_train_leoe_scaled_mn_df=pd.DataFrame(x_train_leoe_scaled_mn, columns=numerical_columns,index=x_train_leoe.index)
print(type(x_train_leoe_scaled_mn_df))
x_train_leoe_scaled_mn_df.head()
#converts into dataframe

<class 'pandas.core.frame.DataFrame'>


,temperature,rainfall
566853,0.722022,0.874873
382311,0.387961,0.087986
241519,0.602303,0.610346
719220,0.180321,0.134987
905718,0.129336,0.707797


In [70]:
x_test_leoe_scaled_mn_df=pd.DataFrame(x_test_leoe_scaled_mn, columns=numerical_columns,index=x_test_leoe.index)
print(type(x_test_leoe_scaled_mn_df))
x_test_leoe_scaled_mn_df.head()
#converts into dataframe

<class 'pandas.core.frame.DataFrame'>


,temperature,rainfall
987231,0.355034,0.683173
79954,0.322835,0.845118
567130,0.360804,0.780092
500891,0.075807,0.115129
55399,0.136115,0.456143


In [71]:
#join the scaled values with other columns - for this we use concat method
final_train_leoe_scaled_mn=pd.concat([x_train_leoe_scaled_df, x_train_leoe.drop(columns=numerical_columns)], axis=1)
final_train_leoe_scaled_mn.head()

,temperature,rainfall,Region,soil_type,Crop,Fertilizer_Used,Irrigation_Used,weather_condition,Days_to_Harvest
566853,0.767268,1.299211,4.652371,4.646157,4.652244,True,True,4.647582,79
382311,-0.389472,-1.426747,4.648410,4.645563,4.652300,False,False,4.648543,88
241519,0.352719,0.382829,4.648391,4.647616,4.654280,False,False,4.647596,120
719220,-1.108459,-1.263925,4.652398,4.654132,4.649399,False,True,4.647603,101
905718,-1.285003,0.720423,4.645159,4.650633,4.652075,True,False,4.648528,88


In [72]:
#join the scaled values with other columns - for this we use concat method
final_test_leoe_scaled_mn=pd.concat([x_test_leoe_scaled_df, x_test_leoe.drop(columns=numerical_columns)], axis=1)
final_test_leoe_scaled_mn.head()

,temperature,rainfall,Region,soil_type,Crop,Fertilizer_Used,Irrigation_Used,weather_condition,Days_to_Harvest
987231,-0.503488,0.635117,4.650130,4.647616,4.649386,False,False,4.650929,120
79954,-0.614983,1.196133,4.652389,4.650044,4.649386,False,False,4.648529,78
567130,-0.483508,0.970870,4.652389,4.646184,4.645186,True,True,4.648529,140
500891,-1.470356,-1.332717,4.650130,4.650044,4.649386,False,True,4.650929,96
55399,-1.261531,-0.151367,4.645160,4.647616,4.652076,False,True,4.647596,65


In [73]:
final_train_leoe_scaled_mn["split"]="train"
final_test_leoe_scaled_mn["split"]="test"

In [74]:
final_leoe_scaled_mn=pd.concat([final_train_leoe_scaled_mn, final_test_leoe_scaled_mn],axis=0)

In [75]:
final_leoe_scaled_mn = final_leoe_scaled_mn.reset_index(drop=True)


final_df3 = pd.concat([final_leoe_scaled_mn, final_y], axis=1)


In [76]:
final_leoe_scaled_mn.to_csv("datasets/agriyield/final_leoe_scaled_mn.csv",index="false")

In [77]:
df_loaded3=pd.read_csv("datasets/agriyield/final_leoe_scaled_mn.csv")

In [78]:
train_loaded_df4=df_loaded3[df_loaded3["split"]=="train"].drop(columns=["split"])
test_loaded_df4=df_loaded3[df_loaded3["split"]=="test"].drop(columns=["split"])

In [79]:
train_loaded_df.dtypes


Unnamed: 0                   int64
temperature                float64
rainfall                   float64
Fertilizer_Used               bool
Irrigation_Used               bool
Days_to_Harvest              int64
soil_type_Clay                bool
soil_type_Loam                bool
soil_type_Peaty               bool
soil_type_Sandy               bool
soil_type_Silt                bool
weather_condition_Rainy       bool
weather_condition_Sunny       bool
Crop_Cotton                   bool
Crop_Maize                    bool
Crop_Rice                     bool
Crop_Soybean                  bool
Crop_Wheat                    bool
Region_North                  bool
Region_South                  bool
Region_West                   bool
dtype: object

In [80]:
print(train_loaded_df.dtypes)
print(train_loaded_df.isnull().sum())
print(train_loaded_df.shape, y_train.shape)


Unnamed: 0                   int64
temperature                float64
rainfall                   float64
Fertilizer_Used               bool
Irrigation_Used               bool
Days_to_Harvest              int64
soil_type_Clay                bool
soil_type_Loam                bool
soil_type_Peaty               bool
soil_type_Sandy               bool
soil_type_Silt                bool
weather_condition_Rainy       bool
weather_condition_Sunny       bool
Crop_Cotton                   bool
Crop_Maize                    bool
Crop_Rice                     bool
Crop_Soybean                  bool
Crop_Wheat                    bool
Region_North                  bool
Region_South                  bool
Region_West                   bool
dtype: object
Unnamed: 0                 0
temperature                0
rainfall                   0
Fertilizer_Used            0
Irrigation_Used            0
Days_to_Harvest            0
soil_type_Clay             0
soil_type_Loam             0
soil_type_Peaty    

In [81]:
y_train = y_train.astype(float)


In [82]:
from sklearn.ensemble import RandomForestRegressor

# Optimized parameters
model_se = RandomForestRegressor(
    n_estimators=50,          # Reduce number of trees (50 instead of 100)
    max_depth=10,             # Limit depth to prevent overfitting and reduce time
    min_samples_split=5,      # Minimum samples per split
    min_samples_leaf=2,       # Minimum samples per leaf
    random_state=42,
    criterion="squared_error",
    n_jobs=-1                 # Use all CPU cores for parallel processing
)

model_ae = RandomForestRegressor(
    n_estimators=50,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    criterion="absolute_error",
    n_jobs=-1
)



model_se.fit(train_loaded_df,y_train)




,n_estimators,50
,criterion,'squared_error'
,max_depth,10
,min_samples_split,5
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [83]:
y_pred=model_se.predict(test_loaded_df)

In [84]:
from sklearn.metrics import mean_squared_error
test_mse=mean_squared_error(y_true=y_test,y_pred=y_pred)
print(test_mse)

0.251686848458138


In [85]:
from sklearn.tree import DecisionTreeRegressor

model_se1 = DecisionTreeRegressor(
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    criterion="squared_error"
)

model_ae = DecisionTreeRegressor(
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    criterion="absolute_error"
)

model_se1.fit(train_loaded_df, y_train)



,criterion,'squared_error'
,splitter,'best'
,max_depth,10
,min_samples_split,5
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [86]:
y_pred1=model_se1.predict(test_loaded_df)

In [87]:
from sklearn.metrics import mean_squared_error
test_mse1=mean_squared_error(y_true=y_test,y_pred=y_pred1)
print(test_mse1)

0.253746493451723


In [88]:
from sklearn.ensemble import GradientBoostingRegressor

model_gbr = GradientBoostingRegressor(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3,
    min_samples_split=20,
    min_samples_leaf=10,
    subsample=0.8,
    max_features='sqrt',
    random_state=42
)

model_gbr.fit(train_loaded_df.astype('float32'), y_train.astype('float32'))



,loss,'squared_error'
,learning_rate,0.1
,n_estimators,50
,subsample,0.8
,criterion,'friedman_mse'
,min_samples_split,20
,min_samples_leaf,10
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [89]:
y_pred2 = model_gbr.predict(test_loaded_df.astype('float32'))


In [90]:
test_mse2 = mean_squared_error(y_true=y_test, y_pred=y_pred2)
print(test_mse2)

0.3900361089434418


In [91]:
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Base estimator
base_estimator = DecisionTreeRegressor(max_depth=8, min_samples_split=10, min_samples_leaf=5, random_state=42)

# Bagging Regressor (use 'estimator' instead of 'base_estimator')
model_bagging = BaggingRegressor(
    estimator=base_estimator,
    n_estimators=50,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

# Fit model
model_bagging.fit(train_loaded_df.astype('float32'), y_train.astype('float32'))

# Predict
y_pred_bagging = model_bagging.predict(test_loaded_df.astype('float32'))

# Evaluation
mse = mean_squared_error(y_test, y_pred_bagging)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_bagging)

print("Bagging Regressor MSE:", mse)
print("Bagging Regressor RMSE:", rmse)
print("Bagging Regressor R²:", r2)


Bagging Regressor MSE: 0.35336615863772053
Bagging Regressor RMSE: 0.5944460939712873
Bagging Regressor R²: 0.8774288301957189


In [92]:
from xgboost import XGBRegressor

model_xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    predictor='cpu_predictor'
)

model_xgb.fit(
    train_loaded_df.astype('float32'),
    y_train.astype('float32'),
    eval_set=[(test_loaded_df.astype('float32'), y_test.astype('float32'))],
    verbose=False
)

y_pred3 = model_xgb.predict(test_loaded_df.astype('float32'))



c:\Users\Vaishnavi\OneDrive\Desktop\info\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:28:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [93]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# Create the Linear Regression model
model_lr = LinearRegression()

# Fit the model on training data
model_lr.fit(train_loaded_df, y_train)

# Predict on test data
y_pred_lr = model_lr.predict(test_loaded_df)

# Evaluate using Mean Squared Error (MSE) and Root MSE
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)

print("Linear Regression MSE:", mse_lr)
print("Linear Regression RMSE:", rmse_lr)


Linear Regression MSE: 0.25077652388905197
Linear Regression RMSE: 0.5007759218343589


In [94]:
#ohe standard scalar
from sklearn.ensemble import RandomForestRegressor

# Optimized parameters
model_se = RandomForestRegressor(
    n_estimators=50,          # Reduce number of trees (50 instead of 100)
    max_depth=10,             # Limit depth to prevent overfitting and reduce time
    min_samples_split=5,      # Minimum samples per split
    min_samples_leaf=2,       # Minimum samples per leaf
    random_state=42,
    criterion="squared_error",
    n_jobs=-1                 # Use all CPU cores for parallel processing
)

model_ae = RandomForestRegressor(
    n_estimators=50,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    criterion="absolute_error",
    n_jobs=-1
)



model_se.fit(train_loaded_df2,y_train)




,n_estimators,50
,criterion,'squared_error'
,max_depth,10
,min_samples_split,5
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [95]:
y_pred=model_se.predict(test_loaded_df)

In [96]:
from sklearn.metrics import mean_squared_error
test_mse_df2=mean_squared_error(y_true=y_test,y_pred=y_pred)
print(test_mse_df2)

0.25177783367061257


In [97]:
from sklearn.ensemble import GradientBoostingRegressor

model_gbr = GradientBoostingRegressor(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3,
    min_samples_split=20,
    min_samples_leaf=10,
    subsample=0.8,
    max_features='sqrt',
    random_state=42
)

model_gbr.fit(train_loaded_df2.astype('float32'), y_train.astype('float32'))
y_pred5= model_gbr.predict(test_loaded_df2.astype('float32'))




In [98]:
test_mse5= mean_squared_error(y_true=y_test, y_pred=y_pred5)
print(test_mse5)

0.386196545814975


In [99]:
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Base estimator
base_estimator = DecisionTreeRegressor(max_depth=8, min_samples_split=10, min_samples_leaf=5, random_state=42)

# Bagging Regressor (use 'estimator' instead of 'base_estimator')
model_bagging = BaggingRegressor(
    estimator=base_estimator,
    n_estimators=50,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

# Fit model
model_bagging.fit(train_loaded_df2.astype('float32'), y_train.astype('float32'))

# Predict
y_pred_bagging = model_bagging.predict(test_loaded_df2.astype('float32'))

# Evaluation
mse = mean_squared_error(y_test, y_pred_bagging)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_bagging)

print("Bagging Regressor MSE:", mse)
print("Bagging Regressor RMSE:", rmse)
print("Bagging Regressor R²:", r2)


Bagging Regressor MSE: 0.3515742486693126
Bagging Regressor RMSE: 0.5929369685466682
Bagging Regressor R²: 0.8780503851908505


In [100]:
from xgboost import XGBRegressor

model_xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    predictor='cpu_predictor'
)

model_xgb.fit(
    train_loaded_df2.astype('float32'),
    y_train.astype('float32'),
    eval_set=[(test_loaded_df2.astype('float32'), y_test.astype('float32'))],
    verbose=False
)

y_pred6 = model_xgb.predict(test_loaded_df2.astype('float32'))



c:\Users\Vaishnavi\OneDrive\Desktop\info\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:31:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [101]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# Create the Linear Regression model
model_lr = LinearRegression()

# Fit the model on training data
model_lr.fit(train_loaded_df2, y_train)

# Predict on test data
y_pred_lr = model_lr.predict(test_loaded_df2)

# Evaluate using Mean Squared Error (MSE) and Root MSE
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)

print("Linear Regression MSE:", mse_lr)
print("Linear Regression RMSE:", rmse_lr)


Linear Regression MSE: 0.2507707447622579
Linear Regression RMSE: 0.5007701516287267


In [102]:
#leoe standard scalar
from sklearn.ensemble import RandomForestRegressor

# Optimized parameters
model_se = RandomForestRegressor(
    n_estimators=50,          # Reduce number of trees (50 instead of 100)
    max_depth=10,             # Limit depth to prevent overfitting and reduce time
    min_samples_split=5,      # Minimum samples per split
    min_samples_leaf=2,       # Minimum samples per leaf
    random_state=42,
    criterion="squared_error",
    n_jobs=-1                 # Use all CPU cores for parallel processing
)

model_ae = RandomForestRegressor(
    n_estimators=50,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    criterion="absolute_error",
    n_jobs=-1
)



model_se.fit(train_loaded_df3,y_train)
y_pred9=model_se.predict(test_loaded_df3)


In [103]:
from sklearn.metrics import mean_squared_error
test_mse10=mean_squared_error(y_true=y_test,y_pred=y_pred9)
print(test_mse10)

0.9830056512444396


In [104]:
from sklearn.tree import DecisionTreeRegressor

model_se1 = DecisionTreeRegressor(
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    criterion="squared_error"
)

model_ae = DecisionTreeRegressor(
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    criterion="absolute_error"
)

model_se1.fit(train_loaded_df3, y_train)
y_pred11=model_se1.predict(test_loaded_df3)


In [105]:
from sklearn.metrics import mean_squared_error
test_mse11=mean_squared_error(y_true=y_test,y_pred=y_pred11)
print(test_mse11)

1.0807607805040753


In [106]:
from sklearn.ensemble import GradientBoostingRegressor

model_gbr = GradientBoostingRegressor(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3,
    min_samples_split=20,
    min_samples_leaf=10,
    subsample=0.8,
    max_features='sqrt',
    random_state=42
)

model_gbr.fit(train_loaded_df3.astype('float32'), y_train.astype('float32'))
y_pred12 = model_gbr.predict(test_loaded_df3.astype('float32'))



In [107]:
test_mse12 = mean_squared_error(y_true=y_test, y_pred=y_pred12)
print(test_mse12)

0.4982353544078456


In [108]:
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Base estimator
base_estimator = DecisionTreeRegressor(max_depth=8, min_samples_split=10, min_samples_leaf=5, random_state=42)

# Bagging Regressor (use 'estimator' instead of 'base_estimator')
model_bagging = BaggingRegressor(
    estimator=base_estimator,
    n_estimators=50,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

# Fit model
model_bagging.fit(train_loaded_df3.astype('float32'), y_train.astype('float32'))

# Predict
y_pred_bagging = model_bagging.predict(test_loaded_df3.astype('float32'))

# Evaluation
mse = mean_squared_error(y_test, y_pred_bagging)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_bagging)

print("Bagging Regressor MSE:", mse)
print("Bagging Regressor RMSE:", rmse)
print("Bagging Regressor R²:", r2)


Bagging Regressor MSE: 1.0188697699948337
Bagging Regressor RMSE: 1.0093907915147797
Bagging Regressor R²: 0.6465873810103031


In [109]:
from xgboost import XGBRegressor

model_xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    predictor='cpu_predictor'
)

model_xgb.fit(
    train_loaded_df3.astype('float32'),
    y_train.astype('float32'),
    eval_set=[(test_loaded_df3.astype('float32'), y_test.astype('float32'))],
    verbose=False
)

y_pred3 = model_xgb.predict(test_loaded_df3.astype('float32'))



c:\Users\Vaishnavi\OneDrive\Desktop\info\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:35:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [110]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# Create the Linear Regression model
model_lr = LinearRegression()

# Fit the model on training data
model_lr.fit(train_loaded_df3, y_train)

# Predict on test data
y_pred_lr = model_lr.predict(test_loaded_df3)

# Evaluate using Mean Squared Error (MSE) and Root MSE
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)

print("Linear Regression MSE:", mse_lr)
print("Linear Regression RMSE:", rmse_lr)


Linear Regression MSE: 0.250772965278298
Linear Regression RMSE: 0.5007723687248509


In [111]:
#leoe standard scalar_mn
from sklearn.ensemble import RandomForestRegressor

# Optimized parameters
model_se = RandomForestRegressor(
    n_estimators=50,          # Reduce number of trees (50 instead of 100)
    max_depth=10,             # Limit depth to prevent overfitting and reduce time
    min_samples_split=5,      # Minimum samples per split
    min_samples_leaf=2,       # Minimum samples per leaf
    random_state=42,
    criterion="squared_error",
    n_jobs=-1                 # Use all CPU cores for parallel processing
)

model_ae = RandomForestRegressor(
    n_estimators=50,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    criterion="absolute_error",
    n_jobs=-1
)



model_se.fit(train_loaded_df4,y_train)
y_pred9=model_se.predict(test_loaded_df4)


In [112]:
#leoe minmaxscalar
y_pred=model_se.predict(test_loaded_df4)

In [113]:
from sklearn.metrics import mean_squared_error
test_mse_df12=mean_squared_error(y_true=y_test,y_pred=y_pred)
print(test_mse_df12)

0.9830056512444396


In [114]:
from sklearn.ensemble import GradientBoostingRegressor

model_gbr = GradientBoostingRegressor(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3,
    min_samples_split=20,
    min_samples_leaf=10,
    subsample=0.8,
    max_features='sqrt',
    random_state=42
)

model_gbr.fit(train_loaded_df3.astype('float32'), y_train.astype('float32'))
y_pred5= model_gbr.predict(test_loaded_df3.astype('float32'))




In [115]:
test_mse14= mean_squared_error(y_true=y_test, y_pred=y_pred5)
print(test_mse14)

0.4982353544078456


In [116]:
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Base estimator
base_estimator = DecisionTreeRegressor(max_depth=8, min_samples_split=10, min_samples_leaf=5, random_state=42)

# Bagging Regressor (use 'estimator' instead of 'base_estimator')
model_bagging = BaggingRegressor(
    estimator=base_estimator,
    n_estimators=50,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

# Fit model
model_bagging.fit(train_loaded_df3.astype('float32'), y_train.astype('float32'))

# Predict
y_pred_bagging = model_bagging.predict(test_loaded_df3.astype('float32'))

# Evaluation
mse = mean_squared_error(y_test, y_pred_bagging)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_bagging)

print("Bagging Regressor MSE:", mse)
print("Bagging Regressor RMSE:", rmse)
print("Bagging Regressor R²:", r2)


Bagging Regressor MSE: 1.0188697699948337
Bagging Regressor RMSE: 1.0093907915147797
Bagging Regressor R²: 0.6465873810103031


In [117]:
from xgboost import XGBRegressor

model_xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    predictor='cpu_predictor'
)

model_xgb.fit(
    train_loaded_df3.astype('float32'),
    y_train.astype('float32'),
    eval_set=[(test_loaded_df3.astype('float32'), y_test.astype('float32'))],
    verbose=False
)

y_pred6 = model_xgb.predict(test_loaded_df3.astype('float32'))



c:\Users\Vaishnavi\OneDrive\Desktop\info\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:38:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [118]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# Create the Linear Regression model
model_lr = LinearRegression()

# Fit the model on training data
model_lr.fit(train_loaded_df3, y_train)

# Predict on test data
y_pred_lr = model_lr.predict(test_loaded_df3)

# Evaluate using Mean Squared Error (MSE) and Root MSE
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)

print("Linear Regression MSE:", mse_lr)
print("Linear Regression RMSE:", rmse_lr)


Linear Regression MSE: 0.250772965278298
Linear Regression RMSE: 0.5007723687248509


In [119]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [120]:
categorical_features=["soil_type","weather_condition","crop"]
numerical_features=["temperature","rainfall"]


In [121]:
preprocessor = ColumnTransformer(transformers=[
    ("encode", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
     ["soil_type","weather_condition","Crop"]),
    ("scale", MinMaxScaler(), numerical_features)
])

In [122]:
import pandas as pd

# Correct sample for inference
sample = pd.DataFrame([{
    "Region": "North",
    "soil_type": "Sandy",
    "Crop": "Corn",
    "rainfall": 550,
    "temperature": 30,
    "Fertilizer_Used": 50,
    "Irrigation_Used": 30,
    "weather_condition": "Sunny",
    "Days_to_Harvest": 120
}])


In [123]:
pipeline=Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model",RandomForestRegressor(
    n_estimators=50,          # Reduce number of trees (50 instead of 100)
    max_depth=10,             # Limit depth to prevent overfitting and reduce time
    min_samples_split=5,      # Minimum samples per split
    min_samples_leaf=2,       # Minimum samples per leaf
    random_state=42,
    criterion="squared_error",
    n_jobs=-1                 # Use all CPU cores for parallel processing
) )
])

In [124]:
print(x_train.shape)
print(y_train.shape)


(800000, 9)
(800000,)


In [125]:
print(x_train.columns.difference(sample.columns))


Index([], dtype='object')


In [126]:
pipeline.fit(x_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('encode', ...), ('scale', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [127]:
x_train.columns

Index(['Region', 'soil_type', 'Crop', 'rainfall', 'temperature',
       'Fertilizer_Used', 'Irrigation_Used', 'weather_condition',
       'Days_to_Harvest'],
      dtype='object')

In [128]:
prediction=pipeline.predict(sample)
prediction[0]

c:\Users\Vaishnavi\OneDrive\Desktop\info\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


np.float64(4.724907069578561)

In [129]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression



In [130]:
sample = pd.DataFrame([{
    "Region": "North",
    "soil_type": "Sandy",
    "Crop": "Corn",
    "rainfall": 550,
    "temperature": 30,
    "Fertilizer_Used": 50,
    "Irrigation_Used": 30,
    "weather_condition": "Sunny",
    "Days_to_Harvest": 120
}])


In [131]:
categorical_features1 = ["Region", "soil_type", "Crop", "weather_condition"]
numerical_features1 = ["rainfall", "temperature", "Fertilizer_Used", "Irrigation_Used", "Days_to_Harvest"]


In [132]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features1),
        ("num", StandardScaler(), numerical_features1)
    ]
)

In [133]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

In [134]:
pipeline.fit(x_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [135]:
prediction = pipeline.predict(sample)
print("Prediction:", prediction[0])

Prediction: 114.29946781769472


In [139]:
import joblib

# Save the pipeline to a .pkl file
joblib.dump(pipeline, "crop_yield_pipeline.pkl")

print("Pipeline saved successfully!")

# Later, to load the pipeline back
loaded_pipeline = joblib.load("crop_yield_pipeline.pkl")

# Test with the same sample
prediction = loaded_pipeline.predict(sample)
print("Prediction after loading:", prediction[0])


Pipeline saved successfully!
Prediction after loading: 114.29946781769472


In [136]:
!pip install --upgrade xgboost



   ---------------------------------------- 0.0/56.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/56.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/56.8 MB ? eta -:--:--
   ---------------------------------------- 0.5/56.8 MB 1.3 MB/s eta 0:00:44
    --------------------------------------- 1.0/56.8 MB 1.6 MB/s eta 0:00:36
   - -------------------------------------- 1.6/56.8 MB 1.6 MB/s eta 0:00:34
   - -------------------------------------- 1.8/56.8 MB 1.8 MB/s eta 0:00:31
   - -------------------------------------- 2.6/56.8 MB 2.1 MB/s eta 0:00:26
   -- ------------------------------------- 3.1/56.8 MB 2.1 MB/s eta 0:00:26
   -- ------------------------------------- 3.7/56.8 MB 2.2 MB/s eta 0:00:24
   --- ------------------------------------ 4.5/56.8 MB 2.4 MB/s eta 0:00:22
   --- ------------------------------------ 5.2/56.8 MB 2.5 MB/s eta 0:00:21
   ---- ----------------------------------- 6.3/56.8 MB 2.8 MB/s eta 0:00:18
   ---- ------------

  You can safely remove it manually.


In [137]:
!pip install category_encoders


In [138]:
!pip install pandas